# Sequence Predictions
This notebook converts each review description into a sequence of integers using Tokenizer

These sequences are then padded so they are all equal length

A Recurrent Neural Network is trained on this data to predict the wine variety

## Filter Data

In [1]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder

df = pd.read_csv('Wine Reviews Dataset/winemag-data-130k-v2.csv')

varieties = [
    'Pinot Noir',
    'Chardonnay',
    'Cabernet Sauvignon',
    'Red Blend',
    'Bordeaux-style Red Blend',
    'Riesling',
    'Sauvignon Blanc',
    'Syrah',
    'Rosé',
    'Merlot'
]

# Keep only the rows with the varieties listed above
df = df[df['variety'].isin(varieties)]

# Replace the wine names with [WINE] so the model doesn't just learn the names of the wines
df['description'] = df['description'].replace(varieties, '[WINE]')

# Discard columns we won't be using
df = df.drop(columns=['Unnamed: 0', 'country', 'designation', 'points', 'price', 'province', 'region_1', 'region_2', 'taster_name', 'taster_twitter_handle', 'title', 'winery'])

# Encode labels
le = LabelEncoder()
df['variety'] = le.fit_transform(df['variety'])

df[0:10]

,description,variety
3,"Pineapple rind, lemon pith and orange blossom ...",6
4,"Much like the regular bottling from 2012, this...",4
10,"Soft, supple plum envelopes an oaky structure ...",1
12,"Slightly reduced, this wine offers a chalky, t...",1
14,Building on 150 years and six generations of w...,2
15,Zesty orange peels and apple notes abound in t...,6
20,Ripe aromas of dark berries mingle with ample ...,5
21,"A sleek mix of tart berry, stem and herb, alon...",4
23,This wine from the Geneseo district offers aro...,3
25,Oak and earth intermingle around robust aromas...,4


## Tokenize features and labels

In [2]:
from keras.preprocessing.text import Tokenizer
from keras.utils import pad_sequences
from sklearn.model_selection import train_test_split

y = df['variety']

# Tokenize descriptions
feature_tokenizer = Tokenizer()
feature_tokenizer.fit_on_texts(df['description'])
X = feature_tokenizer.texts_to_sequences(df['description'])

# Add padding to ensure all sequences have the same length
X = pad_sequences(X)

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=12)

2023-12-07 16:54:08.422713: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2023-12-07 16:54:08.422796: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2023-12-07 16:54:08.471895: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2023-12-07 16:54:08.574370: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2023-12-07 16:54:09.662588: W tensorflow/compiler/tf2

## Recurrent Neural Network

In [10]:
from keras.models import Sequential
from keras.layers import Dense, Dropout, Embedding, LSTM, Masking

# Create the neural network model
model = Sequential()
model.add(Embedding(len(feature_tokenizer.word_index) + 1, 128, input_length=X_train.shape[1], mask_zero=True))
model.add(Masking(mask_value=0.0))
model.add(LSTM(64))
# model.add(LSTM(64, return_sequences=True))
# model.add(LSTM(16))
model.add(Dense(32, activation='relu'))
model.add(Dense(16, activation='relu'))
# model.add(Dropout(0.5))
model.add(Dense(len(varieties), activation='softmax'))

# Compile the model
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# Train the model
model.fit(X_train, y_train, epochs=10, batch_size=16, validation_data=(X_test, y_test))

Epoch 1/10
3567/3567 [==============================] - 207s 57ms/step - loss: 1.0636 - accuracy: 0.6269 - val_loss: 0.7608 - val_accuracy: 0.7408
Epoch 2/10
3567/3567 [==============================] - 204s 57ms/step - loss: 0.5856 - accuracy: 0.8058 - val_loss: 0.6353 - val_accuracy: 0.7885
Epoch 3/10
3567/3567 [==============================] - 204s 57ms/step - loss: 0.4267 - accuracy: 0.8585 - val_loss: 0.6341 - val_accuracy: 0.8002
Epoch 4/10
3567/3567 [==============================] - 203s 57ms/step - loss: 0.3110 - accuracy: 0.8977 - val_loss: 0.7010 - val_accuracy: 0.7996
Epoch 5/10
3567/3567 [==============================] - 204s 57ms/step - loss: 0.2267 - accuracy: 0.9254 - val_loss: 0.7522 - val_accuracy: 0.7969
Epoch 6/10
3567/3567 [==============================] - 204s 57ms/step - loss: 0.1644 - accuracy: 0.9464 - val_loss: 0.8552 - val_accuracy: 0.7876
Epoch 7/10
3567/3567 [==============================] - 204s 57ms/step - loss: 0.1189 - accuracy: 0.9618 - val_loss: 1

## Transformer

In [3]:
from simpletransformers.classification import ClassificationModel, ClassificationArgs
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import logging

logging.basicConfig(level=logging.INFO)
transformers_logger = logging.getLogger("transformers")
transformers_logger.setLevel(logging.WARNING)

# Rename columns to fit the simpletransformers parameters
df = df.rename(columns={'description': 'text', 'variety': 'labels'})
# Split the data into training and testing sets
train_df, eval_df = train_test_split(df, test_size=0.2, random_state=12)

# Model configuration
model_args = ClassificationArgs(
    num_train_epochs=10,
    use_multiprocessing=False,
    use_multiprocessing_for_evaluation=False,
)

model = ClassificationModel(
    "bert", "bert-base-cased", num_labels=10, args=model_args
)

# Train the model
model.train_model(train_df)

# Evaluate the model
result, model_outputs, wrong_predictions = model.eval_model(eval_df)

# Make predictions with the model
predictions, raw_outputs = model.predict(eval_df['text'][0:10].tolist())

print(f"Accuracy: {(len(model_outputs) - len(wrong_predictions)) / len(model_outputs)}")

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.weight', 'classifier.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
INFO:simpletransformers.classification.classification_utils: Converting to features started. Cache is not used.
INFO:simpletransformers.classification.classification_utils: Saving features into cached file cache_dir/cached_train_bert_128_10_2


Epoch:   0%|          | 0/10 [00:00<?, ?it/s]

Running Epoch 0 of 10:   0%|          | 0/7133 [00:00<?, ?it/s]

/home/christian/anaconda3/lib/python3.10/site-packages/torch/optim/lr_scheduler.py:136: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  warnings.warn("Detected call of `lr_scheduler.step()` before `optimizer.step()`. "


Running Epoch 1 of 10:   0%|          | 0/7133 [00:00<?, ?it/s]

...

INFO:simpletransformers.classification.classification_model: Training of bert model complete. Saved to outputs/.
INFO:simpletransformers.classification.classification_utils: Converting to features started. Cache is not used.
INFO:simpletransformers.classification.classification_utils: Saving features into cached file cache_dir/cached_dev_bert_128_10_2


Running Evaluation:   0%|          | 0/1784 [00:00<?, ?it/s]

INFO:simpletransformers.classification.classification_model:{'mcc': 0.8100970503857561, 'eval_loss': 1.4090793502805212}
INFO:simpletransformers.classification.classification_utils: Converting to features started. Cache is not used.


  0%|          | 0/2 [00:00<?, ?it/s]

Accuracy: 0.8335085874518051
